In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import sys

# delete before use!!

#import cartopy
#import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import einops
import numpy as np
import torch
import os
import pickle
import random


sys.path.insert(0, "/user/work/yl18410/new_graphnet/graphnet_LPDM_emulator")
sys.path.insert(1, "/user/work/yl18410/new_graphnet/graphnet_LPDM_emulator")
from model.layers.encoder import *
from model.layers.decoder import *
from model.layers.processor import *
from model.layers.graph_net_block import *
from model.data.dataloader_graphnet import *
from model.data.load_data import *
from model.forecast import GraphSatelliteForecaster
from model.loss_functions import *
from general_train_nawid import *

import torch.optim as optim
from sklearn.metrics import mean_squared_error, r2_score
import time
from datetime import datetime
import json
import argparse

import random


In [26]:
# Manual setup for Jupyter
file_name = "parameter_template_train_small.json"
file_path = "/user/work/yl18410/new_graphnet/graphnet_LPDM_emulator/parameter_files/"
path = "/user/work/yl18410/new_graphnet/graphnet_LPDM_emulator/"

# Load the actual dictionary
parameters = load_file(file_name, file_path)

if parameters is None:
    raise ValueError("Could not load parameters. Check your paths!")

model_name = parameters["model_name"]
print(f"Model Name: {model_name}")

# Setup directories
os.makedirs(f"{path}{model_name}", exist_ok=True)
os.makedirs(f"{path}{model_name}/training_imgs", exist_ok=True)

# Set seed for reproducibility
set_reproducibility(parameters)

Model Name: test_run
Using seed: 34
CUDA not available; skipping GPU seed initialization.


34

In [18]:
parameters

{'model_name': 'test_run',
 'train_load_data': {'year': '2015',
  'freq': 100,
  'region': 'BRAZIL',
  'size': 50,
  'verbose': True,
  'met_args': {'met_levels': [3, 15, 21]}},
 'test_load_data': {'year': '2016', 'freq': 300},
 'variables': {'met_variables': {'x_wind': [3, 15],
   'y_wind': [3, 15],
   'upward_air_velocity': [3, 15],
   'atmosphere_boundary_layer_thickness': [],
   'surface_air_pressure': []},
  'static_variables': ['lat_coords', 'lon_coords'],
  'time_deltas': [6]},
 'dataloader_parameters': {'input_transforms': ['clever_transform_3'],
  'output_transforms': ['logv4']},
 'model_parameters': {'num_blocks': 4,
  'node_dim': 64,
  'edge_dim': 64,
  'hidden_layers_processor_node': 2,
  'hidden_layers_processor_edge': 2,
  'hidden_layers_decoder': 1,
  'hidden_dim_processor_node': 16,
  'hidden_dim_processor_edge': 16,
  'hidden_dim_decoder': 16,
  'resolution': 4,
  'output_dim': 1,
  'residuals': False,
  'attention': False},
 'learning_rate': 5e-05,
 'epochs': {'traini

In [19]:
import pickle

# Replace with your actual path
file_path = "/user/work/yl18410/new_graphnet/graphnet_LPDM_emulator/practice_data_2014_6_updated.pkl"

with open(file_path, 'rb') as handle:
    data = pickle.load(handle)

In [7]:
data.keys()

dict_keys(['data', 'test_data', 'grid', 'inputs', 'test_inputs', 'outputs', 'test_outputs', 'names', 'train_info', 'test_info'])

In [20]:
train_data,test_data = data['data'], data['test_data']
inputs, test_inputs = data['inputs'],data['test_inputs']
grid, names = data['grid'], data['names']


In [21]:
# Build Datasets
train_dataset = FootprintsDatasetV3(inputs, train_data.fp_data, input_names=names, **parameters["dataloader_parameters"])
test_dataset = FootprintsDatasetV3(test_inputs, test_data.fp_data, input_names=names, 
                                   test_mode=train_dataset.transform_parameters, **parameters["dataloader_parameters"])

# Build Loaders
train_loader = DataLoader(train_dataset, batch_size=5, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=10)

# Store metadata for plotting
if train_data.dataset_format == "square":
    size = [train_data.size, train_data.size]
else:
    size = train_data.domain_size

image_plots = random.sample(list(range(len(test_inputs))), k=4)
image_dates = np.datetime_as_string(test_data.fp_data_full.time.values[image_plots])

assuming this is a square dataset of size 50 x 50!
{}
{}
init logv4
assuming this is a square dataset of size 50 x 50!
{}
{}
init logv4


In [22]:
feature_dim = np.shape(inputs)[-1]
model = GraphSatelliteForecaster(grid, whole_world=False, feature_dim=feature_dim, aux_dim=0, **parameters["model_parameters"])

if torch.cuda.is_available():
    model.cuda()

criterion = eval(parameters["loss_functions"]["criterion"])
criterion_test = eval(parameters["loss_functions"]["criterion_test"])
optimizer = optim.AdamW(model.parameters(), lr=parameters["learning_rate"])

# Initialize tracking dictionaries
flux_evaluation = ["uniform", "checkerboard_10", "checkerboard_5"]
losses = {"train":[], "test":[], "NMAE_test":[], "MSE_test_transformed":[], "NMAE_test_transformed":[], "accuracy":[], "IoU":[]}
losses.update({f"flux_{f}":{"MAE":[], "R2":[]} for f in flux_evaluation})

in satellite encoder!
set up processor
set up graph
in graph processor!
hello
hello
hello
hello
set up decoder


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
epoch_so_far = 0

# If you want to debug, you can leave ipdb in, but usually, 
# in Jupyter, we just let the error happen to see the traceback.
run_full_training(
    model,parameters, train_loader, test_loader, optimizer, criterion, criterion_test, 
    test_dataset, test_data, device, epoch_so_far, losses, 
    flux_evaluation, image_plots, image_dates, size, path, model_name, parameters["learning_rate"], NMAE_function=NMAE_nans
)

Using device: cpu
{'training': 300, 'visualize': 5, 'patience': 3, 'model_saving': 50}

--- Start Epoch: 0 ---
[0,     0] Loss: 1.426 Time: 0.3s
[0,    10] Loss: 1.330 Time: 2.6s
[0,    20] Loss: 1.209 Time: 4.9s
[0,    30] Loss: 1.137 Time: 7.2s
[0,    40] Loss: 1.118 Time: 9.5s
[0,    50] Loss: 1.089 Time: 11.8s
[0,    60] Loss: 1.054 Time: 14.2s
[0,    70] Loss: 1.035 Time: 16.4s
[0,    80] Loss: 1.009 Time: 18.7s
[0,    90] Loss: 0.979 Time: 21.1s
[0,   100] Loss: 0.955 Time: 23.5s


In [11]:
data.keys()
train_data, test_data,inputs,test_inputs,grid, names,train_info,test_info = data['data'], data['test_data'],data['inputs'], data['test_inputs']. data['grid'], data['train_info'], data['test_info']

TypeError: memoryview: invalid slice key

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#### Load Data
train_load_data = copy.deepcopy(parameters["train_load_data"])
test_load_data = copy.deepcopy(parameters["train_load_data"])
test_load_data.update(parameters["test_load_data"])

data = LoadSquareSatelliteData(**train_load_data, load_everything=True)
test_data = LoadSquareSatelliteData(**test_load_data, load_everything=True)

# Extract inputs
input_variables = parameters["variables"]
inputs, names = get_square_satellite_inputs(data, **input_variables, return_variable_names=True, return_asarray=True)
test_inputs = get_square_satellite_inputs(test_data, **input_variables, return_asarray=True)

# Get the grid
grid, _ = get_grid(data, parameters.get("grid_reference_fp"))

Using device: cpu
---- LOADING FOOTPRINTS
Loading footprint data from /group/chem/acrg/LPDM/fp_NAME_pre20210701/SOUTHAMERICA/*BRAZIL*SOUTHAMERICA_2015*.nc
there was an error opening the dataset. checking if any of the files are in the bad files list
at least one of the files was in the bad files list, opening with workaround
reduced the number of datapoints by frequency 100
Loading 174 footprints
----- Cutting footprints to square of size 50
careful! We had to pad the footprints along the longitude dimension to extract size 50 (0 and 1 idxs on either side) Padding with nan
2 footprints were at least partially filled with nans because they were cutting outside of the footprint file domain (this is 1.15% of samples)
Padding was needed for the following number of footprints along each direction: {'N': 0, 'S': 0, 'E': 2, 'W': 0}

 ---- LOADING MET
Loading meteorology from /group/chem/acrg/met_archive/UM/SOUTHAMERICA/SOUTHAMERICA_Met_2015*.nc
----- Cutting met
calculated wind angle and/or s